In [43]:
import pandas as pd
import numpy as np

## Utility Functions

These utility functions include:
- load_and_preprocess_data: To load the data from the file and select the required columns.
- aggregate_to_1min_intervals: To create 1 minute intervals for OFI calculations for each interval.
- compute_bid_ask_ofi: Returns the summation of a particular level.
- compute_ofi: Handles the number of levels to be included for OFI calculations.

In [44]:
def load_and_preprocess_data(file_path, num_levels=1):
    """
    Load the dataset and preprocess timestamps and relevant columns for OFI calculation.
    
    Args:
        file_path (str): Path to the CSV file.
        num_levels (int): Depth required.
        
    Returns:
        pd.DataFrame: Preprocessed DataFrame with parsed timestamps and relevant columns.
    """
    columns = ['ts_event']
    for level in range(num_levels):
        columns.extend([f'bid_px_{level:02d}', f'bid_sz_{level:02d}', 
                        f'ask_px_{level:02d}', f'ask_sz_{level:02d}'])
    
    # Load the dataset
    df = pd.read_csv(file_path, usecols=columns)
    
    # Convert ts_event to datetime
    df['ts_event'] = pd.to_datetime(df['ts_event'])
    
    # Sort by timestamp to ensure chronological order
    df = df.sort_values('ts_event')
    
    return df

# Aggregate data into 1-minute intervals
def aggregate_to_1min_intervals(df):
    """
    Aggregate LOB updates into 1-minute intervals, taking the last snapshot per interval.
    
    Args:
        df (pd.DataFrame): Preprocessed DataFrame with LOB data.
        
    Returns:
        pd.DataFrame: Aggregated DataFrame with one row per 1-minute interval.
    """
    # Set ts_event as index for resampling
    df = df.set_index('ts_event')
    
    # Resample to 1-minute intervals, taking the last snapshot in each interval
    df_1min = df.resample('1min').last()
    
    # Reset index to make ts_event a column again
    df_1min = df_1min.reset_index()
    
    # Drop rows with missing values (if an interval has no updates)
    df_1min = df_1min.dropna()
    
    return df_1min

def compute_bid_ask_ofi(df, level=0):
    """
    Compute bid and ask OFI using the 3-condition logic at a given LOB level.

    Args:
        df (pd.DataFrame): DataFrame containing bid/ask px/sz and their previous values.
        level (int): LOB level to compute (default: 0).

    Returns:
        np.ndarray: Combined OFI (bid + ask) for the specified level.
    """
    bid_px = df[f'bid_px_{level:02d}']
    bid_sz = df[f'bid_sz_{level:02d}']
    bid_px_prev = df[f'bid_px_prev']
    bid_sz_prev = df[f'bid_sz_prev']

    ask_px = df[f'ask_px_{level:02d}']
    ask_sz = df[f'ask_sz_{level:02d}']
    ask_px_prev = df[f'ask_px_prev']
    ask_sz_prev = df[f'ask_sz_prev']

    # Bid side OFI
    bid_ofi = np.where(
        bid_px > bid_px_prev, bid_sz,
        np.where(bid_px < bid_px_prev, -bid_sz_prev, bid_sz - bid_sz_prev)
    )

    # Ask side OFI
    ask_ofi = np.where(
        ask_px > ask_px_prev, -ask_sz,
        np.where(ask_px < ask_px_prev, ask_sz_prev, ask_sz - ask_sz_prev)
    )

    return bid_ofi + ask_ofi

# Compute Multi-Level OFI
def compute_ofi(df_1min, num_levels=10):
    """
    Compute Multi-Level OFI by summing OFI across multiple levels (0 to num_levels-1).
    
    Args:
        df_1min (pd.DataFrame): Aggregated DataFrame with 1-minute snapshots.
        num_levels (int): Number of LOB levels to consider (default: 10).
        
    Returns:
        pd.DataFrame: DataFrame with timestamps and corresponding Multi-Level OFI values.
    """

    df = df_1min.copy()
    for level in range(num_levels):
        # Shifted (previous) values
        df['bid_px_prev'] = df[f'bid_px_{level:02d}'].shift(1)
        df['bid_sz_prev'] = df[f'bid_sz_{level:02d}'].shift(1)
        df['ask_px_prev'] = df[f'ask_px_{level:02d}'].shift(1)
        df['ask_sz_prev'] = df[f'ask_sz_{level:02d}'].shift(1)

        df[f'level_{level:02d}_ofi'] = compute_bid_ask_ofi(df, level=level)

    return df

## Best-Level OFI

In [45]:
# Best Level OFI computation
def bestLevel(file_path):
    """
    To compute Best-Level OFI.
    
    Args:
        file_path (str): Path to the CSV file.
        
    Returns:
        pd.DataFrame: DataFrame with timestamps and Best-Level OFI values.
    """
    NUM_LEVELS = 1
    # Load and preprocess data
    df = load_and_preprocess_data(file_path, num_levels=1)
    
    # Aggregate to 1-minute intervals
    df_1min = aggregate_to_1min_intervals(df)

    # Compute Best-Level OFI (levels 0 to 9)
    df = compute_ofi(df_1min, num_levels=NUM_LEVELS)

    result = pd.DataFrame({
        'ts_event': df_1min['ts_event'],
        'best_level_ofi': df['level_00_ofi']
    }).dropna()
    
    return result

file_path = "first_25000_rows.csv"
best_level_ofi_df = bestLevel(file_path=file_path)
best_level_ofi_df

,ts_event,best_level_ofi
1,2024-10-21 11:55:00+00:00,-545.0
2,2024-10-21 11:56:00+00:00,-173.0
3,2024-10-21 11:57:00+00:00,202.0
4,2024-10-21 11:58:00+00:00,-8.0
5,2024-10-21 11:59:00+00:00,-150.0
...,...,...
66,2024-10-21 13:00:00+00:00,-100.0
67,2024-10-21 13:01:00+00:00,365.0
68,2024-10-21 13:02:00+00:00,261.0
69,2024-10-21 13:03:00+00:00,-199.0


## Multi-Level OFI

In [46]:
# Main function to orchestrate the computation
def multiLevel(file_path):
    """
    Main function to compute Multi-Level OFI.
    
    Args:
        file_path (str): Path to the CSV file.
        
    Returns:
        pd.DataFrame: DataFrame with timestamps and Multi-Level OFI values.
    """
    NUM_LEVELS = 10
    # Load and preprocess data
    df = load_and_preprocess_data(file_path, num_levels=10)
    
    # Aggregate to 1-minute intervals
    df_1min = aggregate_to_1min_intervals(df)
    
    # Compute Multi-Level OFI (levels 0 to 9)
    df = compute_ofi(df_1min, num_levels=NUM_LEVELS)

    # Create result DataFrame
    multi_level_ofi = np.zeros(len(df_1min))

    for level in range(NUM_LEVELS):
        multi_level_ofi += df[f'level_{level:02d}_ofi']

    result = pd.DataFrame({
        'ts_event': df_1min['ts_event'],
        'multi_level_ofi': multi_level_ofi
    }).dropna()
    
    return result

file_path = "first_25000_rows.csv"
multi_level_ofi_df = multiLevel(file_path)
multi_level_ofi_df

,ts_event,multi_level_ofi
1,2024-10-21 11:55:00+00:00,-1591.0
2,2024-10-21 11:56:00+00:00,396.0
3,2024-10-21 11:57:00+00:00,2043.0
4,2024-10-21 11:58:00+00:00,-773.0
5,2024-10-21 11:59:00+00:00,-1599.0
...,...,...
66,2024-10-21 13:00:00+00:00,-2017.0
67,2024-10-21 13:01:00+00:00,-2945.0
68,2024-10-21 13:02:00+00:00,-388.0
69,2024-10-21 13:03:00+00:00,-965.0


## Integrated OFI

In [47]:
def calc_weights(k = 0.5, num_levels = 10):
    """
    Calculate normalized exponential decay weights for LOB levels.

    This function computes weights for each level of the limit order book (LOB)
    using an exponential decay function: w_m = exp(-k * m), where m is the depth level.
    The weights are then normalized to sum to 1.

    Args:
        k (float): Decay rate controlling how quickly weights decrease with depth. Default is 0.5.
        num_levels (int): Number of LOB levels to compute weights for. Default is 10.

    Returns:
        np.ndarray: Normalized array of weights of length `num_levels`, summing to 1.
    """
    weights = np.zeros(num_levels)
    for level in range(num_levels):
        weights[level] = np.exp(-k * level)
    return weights / np.sum(weights)

# Orchestrate the Integrated OFI computation
def integrated(file_path):
    """
    Main function to compute Integrated Multi-Level OFI.
    
    Args:
        file_path (str): Path to the CSV file.
        
    Returns:
        pd.DataFrame: DataFrame with timestamps and Integrated OFI values.
    """
    NUM_LEVELS = 10
    # Load and preprocess data
    df = load_and_preprocess_data(file_path, num_levels=10)
    
    # Aggregate to 1-minute intervals
    df_1min = aggregate_to_1min_intervals(df)

    # Calculate weights
    weights = calc_weights(num_levels=NUM_LEVELS)
    
    # Compute Integrated OFI (levels 0 to 9)
    df = compute_ofi(df_1min, num_levels=NUM_LEVELS)

    # Create result DataFrame
    integrated_ofi = np.zeros(len(df_1min))

    for level in range(NUM_LEVELS):
        integrated_ofi += (weights[level] * df[f'level_{level:02d}_ofi'])

    result = pd.DataFrame({
        'ts_event': df_1min['ts_event'],
        'integrated_ofi': integrated_ofi
    }).dropna()
    
    result['integrated_ofi'] = result['integrated_ofi'].round(2)
    return result

file_path = "first_25000_rows.csv"
integrated_ofi_df = integrated(file_path)
integrated_ofi_df

,ts_event,integrated_ofi
1,2024-10-21 11:55:00+00:00,-350.09
2,2024-10-21 11:56:00+00:00,17.29
3,2024-10-21 11:57:00+00:00,187.46
4,2024-10-21 11:58:00+00:00,26.68
5,2024-10-21 11:59:00+00:00,-71.30
...,...,...
66,2024-10-21 13:00:00+00:00,-134.79
67,2024-10-21 13:01:00+00:00,-26.27
68,2024-10-21 13:02:00+00:00,-148.47
69,2024-10-21 13:03:00+00:00,-135.61


## Cross-Asset OFI

Did not implement currently due to lack of data (only AAPL data available). I intend to create simulated data for 4 more stocks and implement the cross-asset OFI within the next few hours.